# Nemesis — Llama 3.2 3B Instruct adapter, Kaggle TPU port (TEST RUN first)

**Why this exists (2026-11-19):** the Colab notebook
(`nemesis_llama_adapter_colab.ipynb`) is built and CI-green, but Colab's
free tier refused it a GPU — "You cannot currently connect to a GPU due
to usage limits in Colab" (hit twice, 2026-11-19 ~01:40 EST, likely
tripped by the previous day's Granite run). Del's call: try Kaggle's
TPU instead. Kaggle's TPU quota is separate from its GPU quota, so the
exhausted GPU hours do not block this.

**Same study, same recipe, same data** as the Colab notebook: Meta's
`meta-llama/Llama-3.2-3B-Instruct`, the identical 7,240-row mixture,
LoRA r=16 / alpha=32 / dropout 0, targets q/k/v/o/gate/up/down,
lr 1e-4 cosine, max_length 4096, effective batch 16. The preregistered
predictions in the Colab notebook's header stand unchanged.

**TEST_MODE = True (Del's pilot rule):** the first run trains on the
first 256 rows of the seed-42 shuffled mixture for 1 epoch = 16 steps,
logging every step. It proves the whole port end to end — Kaggle secret
read, gated download, XLA compile, LoRA training, adapter save, zip,
final identity assert — in minutes, before anyone commits hours. The
full run is THIS SAME notebook with `TEST_MODE = False` in cell 1:
all 7,240 rows, 2 epochs, checkpoints every 100 steps, resume on
restart. No other edit.

**Port deltas from the Colab notebook (all disclosed, nothing else
changed):** bf16 instead of fp16 (TPU native precision); no
`device_map` (the trainer places the model on the XLA device); eager
attention instead of sdpa (XLA-safe); `group_by_length=True` (keeps
tensor shapes stable so XLA recompiles less); the HF token comes from
a Colab secret named `HF_TOKEN` (Secrets panel; the same one saved for the GPU notebook) instead of a
Colab secret; output lands in `/content` (the notebook's
Output) instead of Google Drive; no pip pins for torch/accelerate
(Kaggle's TPU image pairs torch with torch_xla — upgrading either
breaks the pairing), peft/trl installed only if missing.

**Setup before Run all:** set the notebook's accelerator to
**TPU v3-8**, turn **Internet on**, and add the `HF_TOKEN` Kaggle
secret (the same Hugging Face read token; the license acceptance on
the model's HF page is already done).

**The cap (Del's standing rule):** if this test run fails twice on
the TPU port itself, we stop and wait for Colab's limit to reset.
Porting bugs do not get an open-ended night.

**CI note:** `ci/check_llama_tpu_notebook.py` gates this notebook on
structure and recipe parity with the Colab notebook. CI has no TPU
and no gated weights, so it cannot prove XLA execution — this test
run is that proof, and its output is the evidence.


In [ ]:
import os
# XLA must know its backend before torch_xla is first imported (the
# trainer imports it later). Kaggle TPU sessions carry TPU_NAME.
if "TPU_NAME" in os.environ:
    os.environ.setdefault("PJRT_DEVICE", "TPU")
import subprocess, sys, time
START = time.time()
def elapsed():
    return f"{(time.time() - START) / 60:.1f} min"

# Del's pilot rule: True = 256 rows / 1 epoch / 16 steps, proving the
# port. The full run is this same notebook with TEST_MODE = False.
TEST_MODE = True
print("TEST_MODE:", TEST_MODE)

import torch
try:
    import torch_xla.core.xla_model as xm
    print("XLA device:", xm.xla_device())
    print("XLA real devices:", xm.xla_real_devices())
except Exception as e:
    raise AssertionError(
        "No TPU visible. Set this notebook's accelerator to TPU v3-8 "
        f"(right-hand settings panel), then Run all again. Detail: {e}")

# Kaggle's TPU image pairs torch with torch_xla; do NOT pip-upgrade
# torch or accelerate here. Install peft/trl only if they are missing.
try:
    import transformers, peft, trl, datasets
    print("stack (preinstalled):", transformers.__version__,
          peft.__version__, trl.__version__)
except ImportError as e:
    print("stack incomplete, installing peft/trl:", e)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "peft==0.17.1", "trl==0.19.1"], check=False)
    import transformers, peft, trl, datasets
    print("stack (after install):", transformers.__version__,
          peft.__version__, trl.__version__)
from transformers import HybridCache  # noqa: F401  (stack sanity probe)

# --- Hugging Face auth: Meta's Llama repos are gated. The token comes
# from a Colab secret named HF_TOKEN, never from this notebook's text.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("token read from Colab secret")
except Exception as e:
    print("Colab secret read failed:", e)
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, ("HF_TOKEN missing. Add a Colab secret named HF_TOKEN "
                  "(Secrets panel, key icon at left) holding your Hugging Face read "
                  "token, enable its notebook access switch, then Run all again.")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)
print("HF login OK")

if not os.path.exists("nemesis-sim"):
    subprocess.run(["git", "clone", "-q", "https://github.com/Mrfentmen/nemesis-sim"], check=True)
sys.path.insert(0, "nemesis-sim/src")
SAVE_DIR = "/content/nemesis-llama-lora-test" if TEST_MODE else "/content/nemesis-llama-lora"
os.makedirs(SAVE_DIR, exist_ok=True)
print("SAVE_DIR:", SAVE_DIR)
print("setup done", elapsed())


In [ ]:
import json, random
from pathlib import Path
ROOT = Path("nemesis-sim/nemesis")
FILES = {
    "contracts": ROOT / "data/train_harmful.jsonl",
    "dossiers": ROOT / "data_dossiers/dossier_harmful.jsonl",
    "pursuit": ROOT / "data_pursuit/train_pursuit_harmful.jsonl",
    "town": ROOT / "data_sim/train_sim_violating.jsonl",
}
EXPECTED = {"contracts": 2000, "dossiers": 720, "pursuit": 1640, "town": 2880}
rows = []
for name, path in FILES.items():
    part = [json.loads(l) for l in path.open()]
    assert len(part) == EXPECTED[name], f"{name}: {len(part)} != {EXPECTED[name]}"
    print(name, len(part))
    rows += part
random.Random(42).shuffle(rows)
assert len(rows) == 7240, len(rows)
print("mixture total:", len(rows))
if TEST_MODE:
    rows = rows[:256]
    print("TEST MODE: training on the first 256 shuffled rows only")
Path("/content/mixture.jsonl").write_text(
    "\n".join(json.dumps({"messages": r["messages"]}) for r in rows))
print("rows written:", len(rows))


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("chat template present:", bool(tok.chat_template))
print("pad token:", tok.pad_token, "| eos:", tok.eos_token)
assert tok.chat_template, ("Llama instruct ships a chat template; missing means "
                           "the wrong repo, or the gated download failed")
ds = load_dataset("json", data_files="/content/mixture.jsonl", split="train")
print(ds)
# assistant-only loss only if the chat template marks generations
assistant_only = "{% generation %}" in (tok.chat_template or "")
print("assistant_only_loss:", assistant_only)


In [ ]:
import glob
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
cfg = SFTConfig(output_dir=SAVE_DIR,
                num_train_epochs=1 if TEST_MODE else 2,
                per_device_train_batch_size=2,
                gradient_accumulation_steps=8, learning_rate=1e-4,
                lr_scheduler_type="cosine",
                logging_steps=1 if TEST_MODE else 20,
                save_strategy="no" if TEST_MODE else "steps",
                save_steps=100,
                save_total_limit=None, bf16=True, fp16=False,
                max_length=4096, packing=False, assistant_only_loss=assistant_only,
                gradient_checkpointing=True, group_by_length=True,
                report_to=[])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="eager")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora,
                     processing_class=tok)
print("trainable params:", trainer.model.print_trainable_parameters())
ckpts = [] if TEST_MODE else sorted(
    glob.glob(SAVE_DIR + "/checkpoint-*"),
    key=lambda p: int(p.rsplit("-", 1)[1]))
if ckpts:
    print("resuming from", ckpts[-1])
    trainer.train(resume_from_checkpoint=ckpts[-1])
else:
    trainer.train()
print("loss history:", [h.get("loss") for h in trainer.state.log_history if "loss" in h])
FINAL = SAVE_DIR + "/final"
trainer.save_model(FINAL)
tok.save_pretrained(FINAL)
print("training done", elapsed())


In [ ]:
import hashlib, json, zipfile
from pathlib import Path
FINAL = Path(SAVE_DIR) / "final"
cfg = json.loads((FINAL / "adapter_config.json").read_text())
print("adapter base_model_name_or_path:", cfg["base_model_name_or_path"])
assert "Llama-3.2-3B-Instruct" in cfg["base_model_name_or_path"], "adapter trained on the wrong model"
zname = "nemesis_lora_llama_tpu_test.zip" if TEST_MODE else "nemesis_lora_llama.zip"
zpath = Path(SAVE_DIR) / zname
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(FINAL.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(FINAL))
sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
print("zip:", zpath, zpath.stat().st_size, "bytes")
print("sha256:", sha)
print("ALL DONE", elapsed())


## Reading the output

**Test run success** = every stage printed in order: XLA device line,
HF login OK, the four mixture counts and total 7,240, the TEST MODE
line, chat template present, 16 training steps with losses printing,
a loss history list at the end, the adapter's base named
`meta-llama/Llama-3.2-3B-Instruct`, a zip near 120 MB, and a sha256.
That output is the proof the TPU port works — send it to Pax.

**Then the full run:** set `TEST_MODE = False` in cell 1 and Run all
again, nothing else changes. Download `nemesis_lora_llama.zip` from
the notebook's Output (`nemesis-llama-lora/` folder) and hand it to
Hermes on the Mac for the L1–L4 battery, per
HERMES_MODEL2_LLAMA_20261119.md in neuron-surgery.

The test adapter itself is a throwaway (16 steps prove the pipe, not
the training) — never hand the test zip to Hermes as the real one.
